这是**秩和比综合评价法 (Rank-Sum Ratio, RSR)** 的详细解析。

这是一个由中国统计学家田凤调教授提出的方法，它非常独特，结合了**非参数统计**（排名）和**参数统计**（正态分布分析）的优势。

---

### 一、 算法原理
**核心思想**：**“不管你数值差多少，我只看你排第几。”**

1.  **非参数部分**：它不直接用原始数据（避免了单位不同、异常值干扰的问题），而是把所有数据转化成**排名（秩）**。
2.  **参数部分**：算出一个 RSR 值后，利用正态分布变换（Probit变换），通过回归分析对评价对象进行**分档排序**（比如：优、良、中、差）。

**打个比方**：
老师要评价学生的综合素质。有的考了100分，有的考了59分。RSR不关心你具体多少分，只关心你在班里排第几名。最后根据排名的分布规律，划定分数线，决定谁是“三好学生”。

---

### 二、 推导与步骤

假设有 $n$ 个评价对象， $m$ 个评价指标。

#### 1. 编秩 (Ranking)
对每一个指标的数据进行排名。
*   **正向指标**（越大越好）：从小到大排。最小值排第1，最大值排第 $n$。
*   **负向指标**（越小越好）：从大到小排。最大值排第1，最小值排第 $n$。
*   *注意：如果有相同数值，通常取平均秩（例如第2和第3名分数一样，则都记为 2.5）。*

#### 2. 计算秩和比 (Calculate RSR)
对于第 $i$ 个对象，将其所有指标的排名加起来，除以总排名的最大可能值。
$$ RSR_i = \frac{\sum_{j=1}^{m} R_{ij}}{m \times n} $$
*   $R_{ij}$ 是第 $i$ 个对象在第 $j$ 个指标上的排名。
*   $RSR$ 的值在 $0$ 到 $1$ 之间（实际是 $1/n$ 到 $1$）。值越大，综合评价越好。

*(注：到这一步其实已经可以用来简单排序了，但RSR的精髓在于下一步的分档)*

#### 3. 计算概率单位 (Probit)
为了进行分档，我们需要把 RSR 值转化为符合正态分布的变量。
1.  将算出来的 $RSR$ 值从小到大排序。
2.  计算累积频率 $p_i = \frac{R_i - 0.5}{n}$ （$R_i$ 是 RSR 值的排名，$-0.5$ 是修正系数）。
3.  查询标准正态分布表，找到 $p_i$ 对应的 $u$ 值。
4.  计算 Probit 值：$Y = u + 5$ （加5是为了把负数变成正数，方便计算）。

#### 4. 回归方程计算
以 **Probit值** 为自变量 $X$，以 **RSR值** 为因变量 $Y$（注意这里通常反过来，或者看具体教材，但通常我们拟合 $RSR = a + b \times Probit$），进行一元线性回归。
$$ \widehat{RSR} = a + b \times \text{Probit} $$

#### 5. 分档排序
根据回归计算出的 $\widehat{RSR}$，按照标准（如 Prob < 4 为差，4-6 为中，>6 为优）进行分档。

---

### 三、 适用场景
1.  **医疗卫生评价**：RSR在医学统计领域应用极广（如医院科室评价、医疗质量综合评价）。
2.  **异常值较多**：数据波动极大，或者有明显的离群点，用均值方差法不准确时，RSR非常稳健。
3.  **分档定级**：不仅要排名次，还要把对象划分为“优、良、差”几个等级。

---

### 四、 Python 代码框架

这份代码实现了完整的 RSR 流程，包括**编秩、RSR计算、Probit变换、回归拟合**以及**分档展示**。

```python
import pandas as pd
import numpy as np
from scipy.stats import norm, linregress
import matplotlib.pyplot as plt

def rsr_method(data, direction_list, plot=False):
    """
    秩和比综合评价法 (RSR)
    :param data: pandas DataFrame, 原始数据 (n行样本 x m列指标)
    :param direction_list: list, 指标方向 (1:正向/越大越好, 0:负向/越小越好)
    :param plot: bool, 是否绘制回归拟合图
    :return: pandas DataFrame (包含RSR值、拟合值、分档等级)
    """
    df = data.copy().astype(float)
    n, m = df.shape

    # --- 1. 编秩 (Rank) ---
    # 也就是把原始数据变成排名
    rank_df = pd.DataFrame(index=df.index, columns=df.columns)

    for i, col in enumerate(df.columns):
        direction = direction_list[i]
        if direction == 1: # 正向: 值越大，排名越大(越好) -> method='average'处理并列
            rank_df[col] = df[col].rank(method='average', ascending=True)
        else: # 负向: 值越小，排名越大(越好)
            rank_df[col] = df[col].rank(method='average', ascending=False)

    # --- 2. 计算 RSR 值 ---
    # RSR = Sum(Ranks) / (m * n)
    df['Sum_Rank'] = rank_df.sum(axis=1)
    df['RSR'] = df['Sum_Rank'] / (m * n)

    # --- 3. 计算 Probit (概率单位) ---
    # 为了做回归，需要对 RSR 进行排序，并计算累积频率

    # 这里的排序是为了计算频率分布，不是最终排名
    # 将 RSR 提取出来单独处理
    rsr_series = df['RSR'].sort_values()

    # 计算累计频率 p = (Rank_of_RSR - 0.5) / n
    # 注意：这里的 Rank_of_RSR 是 RSR 值本身的排名 (1, 2, ..., n)
    rsr_ranks = np.arange(1, n + 1)
    proportions = (rsr_ranks - 0.5) / n

    # 计算 Probit = NormInv(p) + 5
    probits = norm.ppf(proportions) + 5

    # 将计算结果匹配回原 DataFrame (有点绕，因为要对应索引)
    # 创建一个临时 DF 来存 Probit
    temp_df = pd.DataFrame({
        'RSR': rsr_series.values,
        'Probit': probits
    }, index=rsr_series.index)

    # 合并回去
    df = df.join(temp_df[['Probit']])

    # --- 4. 线性回归拟合 ---
    # 模型: RSR = a + b * Probit
    # 这里的自变量是 Probit (X), 因变量是 RSR (Y)
    slope, intercept, r_value, p_value, std_err = linregress(df['Probit'], df['RSR'])

    print(f"回归方程: RSR = {intercept:.4f} + {slope:.4f} * Probit")
    print(f"拟合优度 R^2: {r_value**2:.4f}")

    # 计算拟合值 (Fitted RSR)
    df['Fitted_RSR'] = intercept + slope * df['Probit']

    # --- 5. 分档排序 (Leveling) ---
    # 常用分档标准 (根据 Probit 值划分):
    # Probit < 4: 差
    # 4 <= Probit < 6: 中
    # 6 <= Probit < 8: 良
    # Probit >= 8: 优 (一般很少有>=8，除非样本量极大)

    def get_level(probit):
        if probit < 4: return '差 (Level 1)'
        elif probit < 6: return '中 (Level 2)'
        elif probit < 8: return '良 (Level 3)'
        else: return '优 (Level 4)'

    df['Level'] = df['Probit'].apply(get_level)
    df['Rank'] = df['Fitted_RSR'].rank(ascending=False) # 按拟合值排名

    # --- 6. 绘图 (可选) ---
    if plot:
        plt.figure(figsize=(8, 5))
        plt.scatter(df['Probit'], df['RSR'], label='Original RSR')
        plt.plot(df['Probit'], df['Fitted_RSR'], color='red', label='Regression Line')
        plt.xlabel('Probit')
        plt.ylabel('RSR')
        plt.title('RSR Distribution Regression')
        plt.legend()
        plt.grid(True)
        plt.show()

    return df[['RSR', 'Probit', 'Fitted_RSR', 'Level', 'Rank']].sort_values(by='Rank')

# ================= 使用示例 =================

if __name__ == "__main__":
    # 场景：评价 7 个科室的医疗质量
    # 指标：治愈率(正), 死亡率(负), 平均住院日(负), 病床周转次(正)
    data = {
        '治愈率(%)': [85, 90, 88, 82, 95, 80, 89],
        '死亡率(%)': [2.1, 1.5, 1.8, 2.5, 1.0, 3.0, 1.6],
        '平均住院日': [12, 10, 11, 14, 9, 15, 10.5],
        '病床周转次': [15, 18, 16, 12, 20, 10, 17]
    }
    df = pd.DataFrame(data, index=['科室A', '科室B', '科室C', '科室D', '科室E', '科室F', '科室G'])

    print("原始数据:\n", df)

    # 定义方向: 治愈率(1), 死亡率(0), 住院日(0), 周转次(1)
    directions = [1, 0, 0, 1]

    # 运行 RSR 模型
    results = rsr_method(df, directions, plot=True)

    print("-" * 30)
    print("RSR 评价结果:")
    print(results)
```

### 代码使用的“修修补补”指南：

1.  **关于 RSR 值的含义**：
    *   `RSR` (实际计算值) 和 `Fitted_RSR` (回归拟合值)。
    *   **排序用哪个？** 一般用 `Fitted_RSR` 进行最终排序，因为它经过了正态平滑，消除了部分随机误差。但如果你的拟合优度 $R^2$ 很低（比如小于 0.6），说明数据不服从正态分布，这时候直接用原始 `RSR` 值排序更靠谱。

2.  **关于编秩方法 (Rank Method)**：
    *   代码中使用了 `method='average'`。
    *   意思是：如果两个人并列第2名，那么他们的排名都是 $(2+3)/2 = 2.5$。
    *   这是统计学上最严谨的做法。不要随意改成 `min` 或 `max`，会影响 RSR 值的准确性。

3.  **分档标准**：
    *   代码里用的是最经典的 Probit 分档：
        *   $<4$ (对应RSR分布的下15.8%)
        *   $4 \sim 6$ (对应中间的68.3%)
        *   $>6$ (对应上15.8%)
    *   *怎么改？* 如果你想分5档，或者你的样本特别多，可以自定义 `get_level` 函数里的阈值。Probit 的中心值是 5。

4.  **权重问题**：
    *   标准的 RSR 是**等权重**的（认为每个指标一样重要）。
    *   *如果要加权重怎么办？* 也就是 **WRSR (加权秩和比)**。
    *   **修改位置**：在计算 `df['RSR']` 那一行。
        *   原代码：`sum(ranks) / (m * n)`
        *   修改后：`sum(ranks * weights) / n` (注意分母变了，分子是秩乘以权重求和)。如果你需要 WRSR，可以在代码里加一个 `weights` 参数，并在 Step 2 修改公式。